## 1. Imports

In [ ]:
from scipy.io.arff import loadarff
import pandas as pd
import numpy as np

## 2. Loading the Dataset

The dataset is in `.arff` format, native to the Weka ecosystem.
`loadarff` returns the data and metadata separately; the metadata is discarded
with `_` since it is not used in this pipeline.

In [ ]:
data, _ = loadarff('supermarket.arff')

df = pd.DataFrame(np.array(data), dtype=str)

print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 3. Removing Department Columns

Columns prefixed with `department` are structural metadata from the original
Weka file. They do not represent purchasable items and have no analytical value
for basket mining. They are identified and removed automatically using Pandas
string filtering, avoiding any hardcoded column names.

In [ ]:
dept_cols = df.filter(like='department').columns
df = df.drop(columns=dept_cols)

print(f"Removed {len(dept_cols)} department columns.")
print(f"Remaining shape: {df.shape}")

## 4. Removing Zero-Support Columns

Columns where every value is `?` indicate items that never appeared in any
transaction — that is, items with support equal to zero. These columns carry
no information and only increase dimensionality unnecessarily.

A boolean mask checks whether **all** values in a column equal `?`. Columns
that satisfy this condition are dropped. An assertion confirms the operation
was successful before proceeding.

In [ ]:
empty_cols = df.columns[(df == '?').all()]
df = df.drop(columns=empty_cols)

print(f"Removed {len(empty_cols)} zero-support columns.")
print(f"Final shape: {df.shape}")

assert not (df == '?').all().any(), "Zero-support columns still present after cleaning."
print("Assertion passed: no zero-support columns remaining.")

## 5. Exporting the Cleaned Dataset

The cleaned DataFrame is exported to CSV for use in Weka. The `index=False`
parameter is required: without it, Pandas writes its integer row index as an
additional column, which Weka would interpret as a valid attribute and include
in rule generation.

In [ ]:
output_path = 'supermarket_clean.csv'
df.to_csv(output_path, index=False)

print(f"File exported: {output_path}")
print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")

## 6. Final Preview

A quick inspection of the cleaned dataset before handing it off to Weka.

In [ ]:
print("Column list:")
print(df.columns.tolist())

print(f"\nMissing values per column (top 10):")
print((df == '?').sum().sort_values(ascending=False).head(10))